# Olist Data Preparation & SQL Layer

This notebook documents the data-preparation work completed for the Olist Review Intelligence project. It covers dataset inspection, missing-value analysis, table relationships, join validation, and preparation of an analysis-ready review dataset for downstream sentiment analysis.

**Scope:** data loading and preparation only. Downstream SQL analytics, sentiment analysis, and semantic search are outside the scope of this notebook.


## 1. Project dataset

The Olist dataset contains multiple related CSV tables covering customers, orders, reviews, products, sellers, payments, and order items. The first step is to inspect the available files and understand their structure before performing joins.

In [ ]:
import os
import glob
import sqlite3
import pandas as pd

DATA_FOLDER = "data"
DB_FILE = "olist.db"

csv_files = sorted(glob.glob(os.path.join(DATA_FOLDER, "*.csv")))

print(f"Found {len(csv_files)} CSV files:\n")
for filepath in csv_files:
    print("-", os.path.basename(filepath))

## 2. Load the datasets

The CSV files are loaded into pandas DataFrames. The same loading approach used in the project is retained, including a `latin-1` fallback for files that may not use UTF-8 encoding.

In [ ]:
def clean_table_name(filepath):
    name = os.path.basename(filepath).replace(".csv", "")
    return name.replace("olist_", "").replace("_dataset", "")

tables = {}

for filepath in csv_files:
    table_name = clean_table_name(filepath)
    
    try:
        df = pd.read_csv(filepath)
    except UnicodeDecodeError:
        df = pd.read_csv(filepath, encoding="latin-1")
    
    tables[table_name] = df
    print(f"{table_name:35} {len(df):>10,} rows x {len(df.columns):>3} columns")

## 3. Inspect table structure

This gives a quick view of the columns available in each table and helps identify the fields required for relationships and downstream review analysis.

In [ ]:
for name, df in tables.items():
    print("\n" + "=" * 70)
    print(f"TABLE: {name}")
    print("=" * 70)
    print("Columns:")
    print(list(df.columns))
    
    display(df.head(3))

## 4. Missing-value analysis

Missing values are checked across all tables. This is particularly important for the review table because review text fields may contain null values and therefore need to be handled carefully by downstream sentiment-analysis work.

In [ ]:
missing_summary = []

for name, df in tables.items():
    null_counts = df.isna().sum()
    null_counts = null_counts[null_counts > 0].sort_values(ascending=False)
    
    for column, count in null_counts.items():
        missing_summary.append({
            "table": name,
            "column": column,
            "missing_count": int(count),
            "missing_percent": round(count / len(df) * 100, 2)
        })

missing_df = pd.DataFrame(missing_summary)

if missing_df.empty:
    print("No missing values were found.")
else:
    display(missing_df.sort_values(["table", "missing_count"], ascending=[True, False]))

### Review-specific missing values

The review table is central to the sentiment-analysis task, so its missing values are inspected separately.

In [ ]:
reviews = tables["order_reviews"]

review_missing = pd.DataFrame({
    "missing_count": reviews.isna().sum(),
    "missing_percent": (reviews.isna().mean() * 100).round(2)
})

display(review_missing[review_missing["missing_count"] > 0].sort_values("missing_count", ascending=False))

## 5. Table relationships

The main relationships used for the review-analysis data layer are:

- `orders.customer_id` → `customers.customer_id`
- `order_reviews.order_id` → `orders.order_id`
- `order_items.order_id` → `orders.order_id`
- `order_items.product_id` → `products.product_id`
- `order_items.seller_id` → `sellers.seller_id`
- `order_payments.order_id` → `orders.order_id`

Understanding these relationships prevents accidental many-to-many joins and helps create a reliable analysis dataset.

## 6. Validate foreign-key relationships

The following checks look for child records whose referenced parent record does not exist. A result of `0` means no unmatched rows were found for that relationship.

In [ ]:
conn = sqlite3.connect(DB_FILE)

checks = {
    "orders -> customers": """
        SELECT COUNT(*) FROM orders o
        LEFT JOIN customers c ON o.customer_id = c.customer_id
        WHERE c.customer_id IS NULL;
    """,
    "order_items -> orders": """
        SELECT COUNT(*) FROM order_items oi
        LEFT JOIN orders o ON oi.order_id = o.order_id
        WHERE o.order_id IS NULL;
    """,
    "order_items -> products": """
        SELECT COUNT(*) FROM order_items oi
        LEFT JOIN products p ON oi.product_id = p.product_id
        WHERE p.product_id IS NULL;
    """,
    "order_reviews -> orders": """
        SELECT COUNT(*) FROM order_reviews r
        LEFT JOIN orders o ON r.order_id = o.order_id
        WHERE o.order_id IS NULL;
    """,
    "order_payments -> orders": """
        SELECT COUNT(*) FROM order_payments op
        LEFT JOIN orders o ON op.order_id = o.order_id
        WHERE o.order_id IS NULL;
    """
}

join_results = []

for label, query in checks.items():
    result = conn.execute(query).fetchone()[0]
    join_results.append({
        "relationship": label,
        "unmatched_rows": result,
        "status": "OK" if result == 0 else "CHECK THIS"
    })

    
join_results_df = pd.DataFrame(join_results)
display(join_results_df)

## 7. Prepare an analysis-ready review dataset

For downstream review/sentiment analysis, review records can be connected to order information through `order_id`. This preserves the review score and review text while adding useful order-level context.

In [ ]:
review_order_query = """
SELECT
    r.review_id,
    r.order_id,
    r.review_score,
    r.review_comment_title,
    r.review_comment_message,
    r.review_creation_date,
    r.review_answer_timestamp,
    o.customer_id,
    o.order_status,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date
FROM order_reviews r
LEFT JOIN orders o
    ON r.order_id = o.order_id;
"""

review_order_df = pd.read_sql_query(review_order_query, conn)
print(f"Analysis-ready review/order dataset: {len(review_order_df):,} rows")
display(review_order_df.head())

## 8. Final preparation checks

Before handing the prepared data to downstream analysis, verify the resulting dataset size, duplicate review IDs, and availability of review text.

In [ ]:
print("Rows:", len(review_order_df))
print("Columns:", len(review_order_df.columns))
    
print("\nDuplicate review IDs:", review_order_df["review_id"].duplicated().sum())
print("Reviews with comment message:", review_order_df["review_comment_message"].notna().sum())
print("Reviews without comment message:", review_order_df["review_comment_message"].isna().sum())

conn.close()

## Conclusion

The Olist data was inspected and prepared for downstream review intelligence work. The preparation included loading the source tables, examining missing values, documenting relationships, validating key joins, and creating a review/order dataset that can be used by subsequent sentiment-analysis work.

**Note:** This notebook documents the data-preparation layer only; it does not include the separate SQL-query, sentiment-analysis, or semantic-search tasks assigned to other project members.